# Sample Order Data Generation

Generates synthetic order data for 5 products across 3 months (Apr–June 2026) with month-specific pricing and varying order volumes.

In [0]:
import random
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
import datetime

# --- Product catalog ---
product_ids   = [1, 2, 3, 4, 5]
product_names = ["tea", "coffee", "soap", "paste", "cream"]

# --- Month-specific prices and order counts ---
month_config = [
    ("2026-04-01", "2026-04-30", 10, 100),   # Apr 2026: price=10, 100 orders
    ("2026-05-01", "2026-05-31", 20,  90),   # May 2026: price=20,  90 orders
    ("2026-06-01", "2026-06-30", 30,  80),   # Jun 2026: price=30,  80 orders
]

# --- Build rows ---
rows = []
order_id = 1
for start_str, end_str, price, num_orders in month_config:
    start_date = datetime.date.fromisoformat(start_str)
    end_date   = datetime.date.fromisoformat(end_str)
    for _ in range(num_orders):
        idx = random.randint(0, 4)  # pick a product at random
        # random day within the month
        delta = (end_date - start_date).days
        order_date = start_date + datetime.timedelta(days=random.randint(0, delta))
        rows.append(Row(
            order_id=order_id,
            product_id=product_ids[idx],
            product_name=product_names[idx],
            product_price=price,
            product_quantity=1,
            order_date=order_date
        ))
        order_id += 1

# --- Create DataFrame ---
schema = StructType([
    StructField("order_id",        IntegerType(), False),
    StructField("product_id",      IntegerType(), False),
    StructField("product_name",    StringType(),  False),
    StructField("product_price",   IntegerType(), False),
    StructField("product_quantity", IntegerType(), False),
    StructField("order_date",       DateType(),    False),
])

orders_df = spark.createDataFrame(rows, schema=schema)

print(f"Total orders generated: {orders_df.count()}")
display(orders_df.orderBy("order_id"))

Total orders generated: 270


order_id,product_id,product_name,product_price,product_quantity,order_date
1,1,tea,10,1,2026-04-18
2,3,soap,10,1,2026-04-18
3,1,tea,10,1,2026-04-02
4,5,cream,10,1,2026-04-25
5,3,soap,10,1,2026-04-03
6,1,tea,10,1,2026-04-19
7,2,coffee,10,1,2026-04-04
8,5,cream,10,1,2026-04-23
9,3,soap,10,1,2026-04-27
10,1,tea,10,1,2026-04-22


In [0]:
%sql
-- Create the 'product' catalog
CREATE CATALOG IF NOT EXISTS product;

-- Create bronze, silver, and gold schemas
CREATE SCHEMA IF NOT EXISTS product.bronze;
CREATE SCHEMA IF NOT EXISTS product.silver;
CREATE SCHEMA IF NOT EXISTS product.gold;

-- Create the orders table in bronze with liquid clustering on order_id and order_date
CREATE TABLE IF NOT EXISTS product.bronze.orders (
    order_id          INT      NOT NULL,
    product_id        INT      NOT NULL,
    product_name      STRING   NOT NULL,
    product_price     INT      NOT NULL,
    product_quantity  INT      NOT NULL,
    order_date        DATE     NOT NULL
)
CLUSTER BY (order_id, order_date);

In [0]:
%sql
-- Verify catalog and schemas
SHOW CATALOGS LIKE 'product';

SHOW SCHEMAS IN product;

-- Verify table schema and liquid clustering
DESCRIBE TABLE product.bronze.orders;

DESCRIBE DETAIL product.bronze.orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,59dc7b78-fd22-442d-86dd-f606d8975569,product.bronze.orders,null,,2026-09-23T10:35:39.062Z,2026-09-23T10:35:41.000Z,List(),"List(order_id, order_date)",0,0,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> aa6fa2eb-60dc-4a20-b0ea-8e44ef1f2761, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-4db0434e-b7f2-4d89-a46c-aae08eaf3d26, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-5f565ab0-ede4-4db5-825d-69a2f2563481)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
# Insert the generated orders into the bronze table
(orders_df.write
    .mode("append")
    .saveAsTable("product.bronze.orders"))

print(f"Inserted {orders_df.count()} rows into product.bronze.orders")

Inserted 270 rows into product.bronze.orders
